In [1]:
import sqlite3

print("SQLite version:", sqlite3.sqlite_version)


SQLite version: 3.50.4


In [2]:
import pandas as pd
import sqlite3

# Dataset path
file_path = r"C:\Users\User\OneDrive\Pictures\Desktop\Shanmukha\Projects\Sales-Intelligence-Analytics\data\raw\sample_-_superstore.xls"

# Load dataset
df = pd.read_excel(file_path)

# Create SQLite database
conn = sqlite3.connect("../data/sales_intelligence.db")

# Store dataframe as SQL table
df.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)

print("Dataset successfully loaded into SQLite!")
print("Rows inserted:", len(df))

Dataset successfully loaded into SQLite!
Rows inserted: 10194


In [3]:
# Check table names
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,sales


In [4]:
# Check number of rows
row_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM sales;",
    conn
)

row_count

,total_rows
0,10194


In [5]:
preview = pd.read_sql_query(
    """
    SELECT *
    FROM sales
    LIMIT 5;
    """,
    conn
)

preview

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,2023-01-03 00:00:00,2023-01-07 00:00:00,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,2023-01-04 00:00:00,2023-01-08 00:00:00,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,2023-01-04 00:00:00,2023-01-08 00:00:00,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,2023-01-04 00:00:00,2023-01-08 00:00:00,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,2023-01-05 00:00:00,2023-01-12 00:00:00,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840


In [6]:
query = """
SELECT
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    SUM(Quantity) AS Total_Quantity,
    COUNT(DISTINCT "Order ID") AS Total_Orders,
    COUNT(DISTINCT "Customer ID") AS Total_Customers,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Percent
FROM sales;
"""

sql_kpis = pd.read_sql_query(query, conn)
sql_kpis

,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Customers,Profit_Margin_Percent
0,2326534.35,292296.81,38654,5111,804,12.56


In [7]:
query = """
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    SUM(Quantity) AS Quantity,
    COUNT(DISTINCT "Order ID") AS Orders
FROM sales
GROUP BY Category
ORDER BY Sales DESC;
"""

sql_category = pd.read_sql_query(query, conn)
sql_category

,Category,Sales,Profit,Quantity,Orders
0,Technology,839893.28,146543.38,7017,1560
1,Furniture,754747.76,19730.00,8369,1819
2,Office Supplies,731893.31,126023.44,23268,3812


In [8]:
query = """
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Percent
FROM sales
GROUP BY Category
ORDER BY Profit_Margin_Percent DESC;
"""

sql_category_margin = pd.read_sql_query(query, conn)
sql_category_margin

,Category,Sales,Profit,Profit_Margin_Percent
0,Technology,839893.28,146543.38,17.45
1,Office Supplies,731893.31,126023.44,17.22
2,Furniture,754747.76,19730.00,2.61


In [9]:
query = """
SELECT
    Region,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    SUM(Quantity) AS Quantity,
    COUNT(DISTINCT "Order ID") AS Orders
FROM sales
GROUP BY Region
ORDER BY Sales DESC;
"""

sql_region = pd.read_sql_query(query, conn)
sql_region

,Region,Sales,Profit,Quantity,Orders
0,West,739813.61,110798.82,12466,1635
1,East,691828.17,94883.26,11159,1475
2,Central,503170.67,39865.31,8820,1179
3,South,391721.91,46749.43,6209,822


In [10]:
query = """
SELECT
    Region,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Percent
FROM sales
GROUP BY Region
ORDER BY Profit_Margin_Percent DESC;
"""

sql_region_margin = pd.read_sql_query(query, conn)
sql_region_margin

,Region,Sales,Profit,Profit_Margin_Percent
0,West,739813.61,110798.82,14.98
1,East,691828.17,94883.26,13.71
2,South,391721.91,46749.43,11.93
3,Central,503170.67,39865.31,7.92


In [11]:
query = """
SELECT
    "Sub-Category",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    SUM(Quantity) AS Quantity,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Percent
FROM sales
GROUP BY "Sub-Category"
ORDER BY Profit DESC;
"""

sql_subcategory = pd.read_sql_query(query, conn)
sql_subcategory

,Sub-Category,Sales,Profit,Quantity,Orders,Profit_Margin_Percent
0,Copiers,150745.29,56093.94,242,70,37.21
1,Phones,331842.64,45050.83,3357,826,13.58
2,Accessories,167380.32,41936.64,2976,718,25.05
3,Paper,79540.54,34511.51,5207,1205,43.39
4,Binders,207354.88,31426.10,6071,1339,15.16
5,Chairs,335768.25,27223.53,2431,591,8.11
6,Storage,224644.55,21285.11,3191,787,9.48
7,Appliances,108213.18,18329.48,1750,459,16.94
8,Furnishings,95598.13,13891.74,3799,919,14.53
9,Envelopes,16528.36,6988.02,914,251,42.28


In [12]:
query = """
SELECT
    "Product ID",
    "Product Name",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    SUM(Quantity) AS Quantity,
    ROUND(AVG(Discount), 3) AS Avg_Discount
FROM sales
GROUP BY "Product ID", "Product Name"
ORDER BY Profit DESC
LIMIT 10;
"""

sql_top_products = pd.read_sql_query(query, conn)
sql_top_products

,Product ID,Product Name,Sales,Profit,Quantity,Avg_Discount
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61599.82,25199.93,20,0.120
1,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,7753.04,31,0.240
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,18839.69,6983.88,38,0.200
3,TEC-CO-10003763,Canon PC1060 Personal Laser Copier,11619.83,4570.93,19,0.150
4,TEC-MA-10001127,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,4094.98,12,0.167
5,TEC-MA-10003979,Ativa V4110MDD Micro-Cut Shredder,7699.89,3772.95,11,0.000
6,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",14299.89,3717.97,11,0.000
7,TEC-AC-10002049,Plantronics Savi W720 Multi-Device Wireless He...,9367.29,3696.28,24,0.057
8,OFF-BI-10001120,Ibico EPK-21 Electric Binding System,15875.92,3345.28,13,0.333
9,TEC-MA-10000045,Zebra ZM400 Thermal Label Printer,6965.70,3343.54,6,0.000


In [13]:
query = """
SELECT
    "Product ID",
    "Product Name",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    SUM(Quantity) AS Quantity,
    ROUND(AVG(Discount), 3) AS Avg_Discount
FROM sales
GROUP BY "Product ID", "Product Name"
HAVING SUM(Profit) < 0
ORDER BY Profit ASC
LIMIT 10;
"""

sql_loss_products = pd.read_sql_query(query, conn)
sql_loss_products

,Product ID,Product Name,Sales,Profit,Quantity,Avg_Discount
0,TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,11099.96,-8879.97,9,0.533
1,TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,16829.90,-4589.97,18,0.400
2,TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,7999.98,-3839.99,4,0.500
3,FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.12,27,0.280
4,FUR-TA-10001889,Bush Advantage Collection Racetrack Conference...,9544.72,-1934.40,33,0.350
5,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17,27,0.450
6,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08,6,0.500
7,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,16656.20,-1299.18,22,0.100
8,FUR-TA-10001950,Balt Solid Wood Round Tables,6518.75,-1201.06,19,0.200
9,FUR-TA-10004289,BoxOffice By Design Rectangular and Half-Moon ...,1706.25,-1148.44,15,0.483


In [14]:
query = """
SELECT
    "Customer ID",
    "Customer Name",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(AVG(Discount), 3) AS Avg_Discount
FROM sales
GROUP BY "Customer ID", "Customer Name"
ORDER BY Sales DESC
LIMIT 10;
"""

sql_top_customers = pd.read_sql_query(query, conn)
sql_top_customers

,Customer ID,Customer Name,Sales,Profit,Orders,Avg_Discount
0,SM-20320,Sean Miller,25043.05,-1980.74,5,0.247
1,TC-20980,Tamara Chand,19052.22,8981.32,5,0.117
2,RB-19360,Raymond Buch,15117.34,6976.10,6,0.094
3,TA-21385,Tom Ashbrook,14595.62,4703.79,4,0.080
4,AB-10105,Adrian Barton,14473.57,5444.81,10,0.240
5,KL-16645,Ken Lonsdale,14175.23,806.85,12,0.200
6,SC-20095,Sanjit Chand,14142.33,5757.41,9,0.064
7,HL-15040,Hunter Lopez,12873.30,5622.43,6,0.018
8,SE-20110,Sanjit Engle,12209.44,2650.68,11,0.111
9,CC-12370,Christopher Conant,12129.07,2177.05,5,0.282


In [15]:
query = """
SELECT
    "Customer ID",
    "Customer Name",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(AVG(Discount), 3) AS Avg_Discount
FROM sales
GROUP BY "Customer ID", "Customer Name"
ORDER BY Profit DESC
LIMIT 10;
"""

sql_profitable_customers = pd.read_sql_query(query, conn)
sql_profitable_customers

,Customer ID,Customer Name,Sales,Profit,Orders,Avg_Discount
0,TC-20980,Tamara Chand,19052.22,8981.32,5,0.117
1,RB-19360,Raymond Buch,15117.34,6976.10,6,0.094
2,SC-20095,Sanjit Chand,14142.33,5757.41,9,0.064
3,HL-15040,Hunter Lopez,12873.30,5622.43,6,0.018
4,AB-10105,Adrian Barton,14473.57,5444.81,10,0.240
5,TA-21385,Tom Ashbrook,14595.62,4703.79,4,0.080
6,CM-12385,Christopher Martinez,8954.02,3899.89,4,0.120
7,KD-16495,Keith Dawkins,8181.26,3038.63,12,0.088
8,AR-10540,Andy Reiter,6608.45,2884.62,6,0.067
9,DR-12940,Daniel Raglin,8350.87,2869.08,8,0.154


In [16]:
query = """
SELECT
    strftime('%Y-%m', "Order Date") AS Month,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders
FROM sales
GROUP BY strftime('%Y-%m', "Order Date")
ORDER BY Month;
"""

sql_monthly = pd.read_sql_query(query, conn)
sql_monthly

,Month,Sales,Profit,Orders
0,2023-01,14518.06,2539.39,33
1,2023-02,4519.89,862.31,28
2,2023-03,56933.91,693.45,72
3,2023-04,28295.34,3488.84,66
4,2023-05,26319.77,3196.39,72
5,2023-06,34669.48,4999.76,67
6,2023-07,33946.39,-841.48,65
7,2023-08,28918.34,5765.22,75
8,2023-09,82670.43,8593.63,134
9,2023-10,32413.34,3469.17,81


In [17]:
query = """
SELECT
    strftime('%Y', "Order Date") AS Year,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders,
    SUM(Quantity) AS Quantity
FROM sales
GROUP BY strftime('%Y', "Order Date")
ORDER BY Year;
"""

sql_yearly = pd.read_sql_query(query, conn)
sql_yearly

,Year,Sales,Profit,Orders,Quantity
0,2023,494040.21,51684.30,995,7813
1,2024,472993.03,62020.97,1053,8086
2,2025,613933.58,82665.20,1340,10018
3,2026,745567.53,95926.35,1723,12737


In [18]:
query = """
SELECT
    "Ship Mode",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders,
    SUM(Quantity) AS Quantity
FROM sales
GROUP BY "Ship Mode"
ORDER BY Sales DESC;
"""

sql_ship_mode = pd.read_sql_query(query, conn)
sql_ship_mode

,Ship Mode,Sales,Profit,Orders,Quantity
0,Standard Class,1378840.55,168161.21,3068,23409
1,Second Class,466671.11,58962.13,982,7546
2,First Class,351750.74,49012.72,795,5724
3,Same Day,129271.96,16160.75,266,1975


In [19]:
query = """
SELECT
    Segment,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders,
    COUNT(DISTINCT "Customer ID") AS Customers,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Percent
FROM sales
GROUP BY Segment
ORDER BY Sales DESC;
"""

sql_segment = pd.read_sql_query(query, conn)
sql_segment

,Segment,Sales,Profit,Orders,Customers,Profit_Margin_Percent
0,Consumer,1170659.79,136371.45,2628,416,11.65
1,Corporate,715806.13,94249.64,1552,239,13.17
2,Home Office,440068.43,61675.73,931,149,14.02


In [20]:
query = """
SELECT
    Discount,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Percent
FROM sales
GROUP BY Discount
ORDER BY Discount;
"""

sql_discount = pd.read_sql_query(query, conn)
sql_discount

,Discount,Sales,Profit,Orders,Profit_Margin_Percent
0,0.00,1105323.79,326718.59,2714,29.56
1,0.10,54952.50,9099.97,91,16.56
2,0.15,27558.52,1418.99,51,5.15
3,0.20,773939.40,91079.95,2440,11.77
4,0.30,104474.07,-10513.45,214,-10.06
5,0.32,14493.46,-2391.14,27,-16.50
6,0.40,116497.76,-23086.37,186,-19.82
7,0.45,5484.97,-2493.11,10,-45.45
8,0.50,58918.54,-20506.43,64,-34.80
9,0.60,7105.94,-6164.35,138,-86.75


In [21]:
query = """
SELECT
    "State/Province",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders
FROM sales
GROUP BY "State/Province"
ORDER BY Sales DESC
LIMIT 10;
"""

sql_top_states = pd.read_sql_query(query, conn)
sql_top_states

,State/Province,Sales,Profit,Orders
0,California,457687.63,76381.39,1021
1,New York,310876.27,74038.55,562
2,Texas,170188.05,-25729.36,487
3,Washington,138641.27,33402.65,256
4,Pennsylvania,116511.91,-15559.96,288
5,Florida,89473.71,-3399.30,200
6,Illinois,80166.10,-12607.89,276
7,Ohio,78258.14,-16971.38,236
8,Michigan,76269.61,24463.19,117
9,Virginia,70636.72,18597.95,115


In [22]:
query = """
SELECT
    "State/Province",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Percent
FROM sales
GROUP BY "State/Province"
ORDER BY Profit DESC
LIMIT 10;
"""

sql_profitable_states = pd.read_sql_query(query, conn)
sql_profitable_states

,State/Province,Sales,Profit,Profit_Margin_Percent
0,California,457687.63,76381.39,16.69
1,New York,310876.27,74038.55,23.82
2,Washington,138641.27,33402.65,24.09
3,Michigan,76269.61,24463.19,32.07
4,Virginia,70636.72,18597.95,26.33
5,Indiana,53555.36,18382.94,34.33
6,Georgia,49095.84,16250.04,33.10
7,Kentucky,36591.75,11199.70,30.61
8,Minnesota,29863.15,10823.19,36.24
9,Delaware,27451.07,9977.37,36.35


In [23]:
query = """
SELECT
    "State/Province",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Percent
FROM sales
GROUP BY "State/Province"
HAVING SUM(Profit) < 0
ORDER BY Profit ASC;
"""

sql_loss_states = pd.read_sql_query(query, conn)
sql_loss_states

,State/Province,Sales,Profit,Profit_Margin_Percent
0,Texas,170188.05,-25729.36,-15.12
1,Ohio,78258.14,-16971.38,-21.69
2,Pennsylvania,116511.91,-15559.96,-13.35
3,Illinois,80166.10,-12607.89,-15.73
4,North Carolina,55603.16,-7490.91,-13.47
5,Colorado,32108.12,-6527.86,-20.33
6,Tennessee,30661.87,-5341.69,-17.42
7,Arizona,35282.00,-3427.92,-9.72
8,Florida,89473.71,-3399.30,-3.80
9,Oregon,17431.15,-1190.47,-6.83


In [24]:
query = """
SELECT
    "Customer ID",
    "Customer Name",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders,
    CASE
        WHEN SUM(Profit) > 5000 THEN 'High Value'
        WHEN SUM(Profit) > 0 THEN 'Profitable'
        ELSE 'Loss Making'
    END AS Customer_Type
FROM sales
GROUP BY "Customer ID", "Customer Name"
ORDER BY Profit DESC;
"""

sql_customer_segments = pd.read_sql_query(query, conn)
sql_customer_segments.head(20)

,Customer ID,Customer Name,Sales,Profit,Orders,Customer_Type
0,TC-20980,Tamara Chand,19052.22,8981.32,5,High Value
1,RB-19360,Raymond Buch,15117.34,6976.10,6,High Value
2,SC-20095,Sanjit Chand,14142.33,5757.41,9,High Value
3,HL-15040,Hunter Lopez,12873.30,5622.43,6,High Value
4,AB-10105,Adrian Barton,14473.57,5444.81,10,High Value
5,TA-21385,Tom Ashbrook,14595.62,4703.79,4,Profitable
6,CM-12385,Christopher Martinez,8954.02,3899.89,4,Profitable
7,KD-16495,Keith Dawkins,8181.26,3038.63,12,Profitable
8,AR-10540,Andy Reiter,6608.45,2884.62,6,Profitable
9,DR-12940,Daniel Raglin,8350.87,2869.08,8,Profitable


In [25]:
query = """
SELECT
    "Product ID",
    "Product Name",
    SUM(Quantity) AS Quantity_Sold,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit
FROM sales
GROUP BY "Product ID", "Product Name"
ORDER BY Quantity_Sold DESC
LIMIT 10;
"""

sql_popular_products = pd.read_sql_query(query, conn)
sql_popular_products

,Product ID,Product Name,Quantity_Sold,Orders,Sales,Profit
0,FUR-FU-10002501,Nu-Dell Executive Frame,84,11,924.74,285.65
1,OFF-BI-10001718,GBC DocuBind P50 Personal Binding Machine,72,14,3753.26,1211.75
2,OFF-BI-10001524,GBC Premium Transparent Covers with Diagonal L...,67,14,780.46,49.51
3,FUR-CH-10002647,"Situations Contoured Folding Chairs, 4/Set",64,15,3641.27,234.23
4,TEC-PH-10001300,iKross Bluetooth Portable Keyboard + Cell Phon...,64,9,1315.66,350.28
5,FUR-TA-10001095,Chromcraft Round Conference Tables,61,13,8209.06,-189.98
6,FUR-FU-10001979,Dana Halogen Swing-Arm Architect Lamp,60,10,2425.42,655.51
7,FUR-CH-10003774,"Global Wood Trimmed Manager's Task Chair, Khaki",59,14,4166.88,-503.12
8,FUR-CH-10004287,SAFCO Arco Folding Chair,59,15,12283.28,1257.52
9,OFF-BI-10004728,Wilson Jones Turn Tabs Binder Tool for Ring Bi...,59,13,212.08,58.51


In [26]:
from pathlib import Path

sql_dir = Path("../sql")
sql_dir.mkdir(exist_ok=True)

sql_file = sql_dir / "business_queries.sql"

print("SQL directory:", sql_dir.resolve())
print("SQL file:", sql_file.resolve())

SQL directory: C:\Users\User\OneDrive\Pictures\Desktop\Shanmukha\Projects\Sales-Intelligence-Analytics\sql
SQL file: C:\Users\User\OneDrive\Pictures\Desktop\Shanmukha\Projects\Sales-Intelligence-Analytics\sql\business_queries.sql


In [27]:
kpi_query = """
-- 1. Overall Business KPIs

SELECT
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    SUM(Quantity) AS Total_Quantity,
    COUNT(DISTINCT "Order ID") AS Total_Orders,
    COUNT(DISTINCT "Customer ID") AS Total_Customers,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin
FROM sales;
"""

In [28]:
category_query = """
-- 2. Category Performance

SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    SUM(Quantity) AS Quantity,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin
FROM sales
GROUP BY Category
ORDER BY Sales DESC;
"""

In [29]:
region_query = """
-- 3. Regional Performance

SELECT
    Region,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin
FROM sales
GROUP BY Region
ORDER BY Sales DESC;
"""

In [30]:
monthly_query = """
-- 4. Monthly Sales and Profit Trend

SELECT
    strftime('%Y-%m', "Order Date") AS Month,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders
FROM sales
GROUP BY strftime('%Y-%m', "Order Date")
ORDER BY Month;
"""

In [31]:
subcategory_query = """
-- 5. Sub-Category Performance

SELECT
    "Sub-Category",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    SUM(Quantity) AS Quantity,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin
FROM sales
GROUP BY "Sub-Category"
ORDER BY Profit DESC;
"""

In [32]:
top_products_query = """
-- 6. Top 10 Profitable Products

SELECT
    "Product ID",
    "Product Name",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    SUM(Quantity) AS Quantity,
    ROUND(AVG(Discount), 3) AS Avg_Discount
FROM sales
GROUP BY "Product ID", "Product Name"
ORDER BY Profit DESC
LIMIT 10;
"""

In [33]:
loss_products_query = """
-- 7. Top 10 Loss-Making Products

SELECT
    "Product ID",
    "Product Name",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    ROUND(AVG(Discount), 3) AS Avg_Discount
FROM sales
GROUP BY "Product ID", "Product Name"
HAVING SUM(Profit) < 0
ORDER BY Profit ASC
LIMIT 10;
"""

In [34]:
customer_query = """
-- 8. Top 10 Customers by Profit

SELECT
    "Customer ID",
    "Customer Name",
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(AVG(Discount), 3) AS Avg_Discount
FROM sales
GROUP BY "Customer ID", "Customer Name"
ORDER BY Profit DESC
LIMIT 10;
"""

In [35]:
discount_query = """
-- 9. Discount and Profitability Analysis

SELECT
    Discount,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin
FROM sales
GROUP BY Discount
ORDER BY Discount;
"""

In [36]:
all_queries = f"""
-- SALES INTELLIGENCE & BUSINESS ANALYTICS
-- Reusable Business Queries
-- Database: sales_intelligence.db

{kpi_query}

{category_query}

{region_query}

{monthly_query}

{subcategory_query}

{top_products_query}

{loss_products_query}

{customer_query}

{discount_query}
"""

In [37]:
sql_file.write_text(
    all_queries,
    encoding="utf-8"
)

print("business_queries.sql created successfully!")
print("Location:", sql_file.resolve())

business_queries.sql created successfully!
Location: C:\Users\User\OneDrive\Pictures\Desktop\Shanmukha\Projects\Sales-Intelligence-Analytics\sql\business_queries.sql


In [38]:
print("File exists:", sql_file.exists())
print("File size:", sql_file.stat().st_size, "bytes")

File exists: True
File size: 2913 bytes


In [39]:
sql_content = sql_file.read_text(encoding="utf-8")

print(sql_content)


-- SALES INTELLIGENCE & BUSINESS ANALYTICS
-- Reusable Business Queries
-- Database: sales_intelligence.db


-- 1. Overall Business KPIs

SELECT
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    SUM(Quantity) AS Total_Quantity,
    COUNT(DISTINCT "Order ID") AS Total_Orders,
    COUNT(DISTINCT "Customer ID") AS Total_Customers,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin
FROM sales;



-- 2. Category Performance

SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    SUM(Quantity) AS Quantity,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin
FROM sales
GROUP BY Category
ORDER BY Sales DESC;



-- 3. Regional Performance

SELECT
    Region,
    ROUND(SUM(Sales), 2) AS Sales,
    ROUND(SUM(Profit), 2) AS Profit,
    COUNT(DISTINCT "Order ID") AS Orders,
    ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin
FROM sales
GROUP

In [40]:
section_count = sql_content.count("-- ")

print("SQL comment/section count:", section_count)

SQL comment/section count: 12


In [41]:
kpi_result = pd.read_sql_query(
    kpi_query,
    conn
)

kpi_result

,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Customers,Profit_Margin
0,2326534.35,292296.81,38654,5111,804,12.56


In [42]:
category_result = pd.read_sql_query(
    category_query,
    conn
)

category_result

,Category,Sales,Profit,Quantity,Orders,Profit_Margin
0,Technology,839893.28,146543.38,7017,1560,17.45
1,Furniture,754747.76,19730.00,8369,1819,2.61
2,Office Supplies,731893.31,126023.44,23268,3812,17.22


In [43]:
region_result = pd.read_sql_query(
    region_query,
    conn
)

region_result

,Region,Sales,Profit,Orders,Profit_Margin
0,West,739813.61,110798.82,1635,14.98
1,East,691828.17,94883.26,1475,13.71
2,Central,503170.67,39865.31,1179,7.92
3,South,391721.91,46749.43,822,11.93


In [44]:
monthly_result = pd.read_sql_query(
    monthly_query,
    conn
)

monthly_result.head(12)

,Month,Sales,Profit,Orders
0,2023-01,14518.06,2539.39,33
1,2023-02,4519.89,862.31,28
2,2023-03,56933.91,693.45,72
3,2023-04,28295.34,3488.84,66
4,2023-05,26319.77,3196.39,72
5,2023-06,34669.48,4999.76,67
6,2023-07,33946.39,-841.48,65
7,2023-08,28918.34,5765.22,75
8,2023-09,82670.43,8593.63,134
9,2023-10,32413.34,3469.17,81


In [45]:
discount_result = pd.read_sql_query(
    discount_query,
    conn
)

discount_result

,Discount,Sales,Profit,Profit_Margin
0,0.00,1105323.79,326718.59,29.56
1,0.10,54952.50,9099.97,16.56
2,0.15,27558.52,1418.99,5.15
3,0.20,773939.40,91079.95,11.77
4,0.30,104474.07,-10513.45,-10.06
5,0.32,14493.46,-2391.14,-16.50
6,0.40,116497.76,-23086.37,-19.82
7,0.45,5484.97,-2493.11,-45.45
8,0.50,58918.54,-20506.43,-34.80
9,0.60,7105.94,-6164.35,-86.75


In [46]:
from pathlib import Path
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


In [47]:
PROJECT_ROOT = Path("..").resolve()

DATA_FILE = PROJECT_ROOT / "data" / "raw" / "sample_-_superstore.xls"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_FILE)

Project root: C:\Users\User\OneDrive\Pictures\Desktop\Shanmukha\Projects\Sales-Intelligence-Analytics
Dataset path: C:\Users\User\OneDrive\Pictures\Desktop\Shanmukha\Projects\Sales-Intelligence-Analytics\data\raw\sample_-_superstore.xls


In [48]:
def load_data(file_path):
    """
    Load the Superstore dataset from an Excel file.
    """
    return pd.read_excel(file_path)

print("load_data() function created.")

load_data() function created.


In [49]:
def calculate_kpis(data):
    """
    Calculate key business KPIs.
    """
    total_sales = data["Sales"].sum()
    total_profit = data["Profit"].sum()
    total_quantity = data["Quantity"].sum()
    total_orders = data["Order ID"].nunique()
    total_customers = data["Customer ID"].nunique()

    profit_margin = (
        total_profit / total_sales * 100
        if total_sales != 0 else 0
    )

    average_order_value = (
        total_sales / total_orders
        if total_orders != 0 else 0
    )

    return {
        "Total Sales": total_sales,
        "Total Profit": total_profit,
        "Total Quantity": total_quantity,
        "Total Orders": total_orders,
        "Total Customers": total_customers,
        "Profit Margin (%)": profit_margin,
        "Average Order Value": average_order_value
    }

print("calculate_kpis() function created.")

calculate_kpis() function created.


In [50]:
kpis = calculate_kpis(df)

for key, value in kpis.items():
    if isinstance(value, float):
        print(f"{key}: {value:,.2f}")
    else:
        print(f"{key}: {value:,}")

Total Sales: 2,326,534.35
Total Profit: 292,296.81
Total Quantity: 38,654
Total Orders: 5,111
Total Customers: 804
Profit Margin (%): 12.56
Average Order Value: 455.20


In [51]:
def category_performance(data):
    """
    Generate category-level sales and profit analysis.
    """
    result = (
        data.groupby("Category")
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("Order ID", "nunique")
        )
        .reset_index()
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Sales", ascending=False)

print("category_performance() function created.")

category_performance() function created.


In [52]:
category_result = category_performance(df)

category_result

,Category,Sales,Profit,Quantity,Orders,Profit Margin (%)
2,Technology,839893.2790,146543.3756,7017,1560,17.45
0,Furniture,754747.7613,19729.9956,8369,1819,2.61
1,Office Supplies,731893.3140,126023.4434,23268,3812,17.22


In [53]:
def region_performance(data):
    """
    Generate region-level sales and profit analysis.
    """
    result = (
        data.groupby("Region")
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("Order ID", "nunique")
        )
        .reset_index()
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Sales", ascending=False)

print("region_performance() function created.")

region_performance() function created.


In [54]:
region_result = region_performance(df)

region_result

,Region,Sales,Profit,Quantity,Orders,Profit Margin (%)
3,West,739813.6085,110798.8170,12466,1635,14.98
1,East,691828.1680,94883.2603,11159,1475,13.71
0,Central,503170.6728,39865.3070,8820,1179,7.92
2,South,391721.9050,46749.4303,6209,822,11.93


In [55]:
def monthly_performance(data):
    """
    Generate monthly sales, profit, and order trends.
    """
    result = (
        data.groupby(data["Order Date"].dt.to_period("M"))
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Orders=("Order ID", "nunique")
        )
        .reset_index()
    )

    result["Order Date"] = result["Order Date"].astype(str)

    return result

In [56]:
monthly_result = monthly_performance(df)

monthly_result.head(12)

,Order Date,Sales,Profit,Orders
0,2023-01,14518.0550,2539.3907,33
1,2023-02,4519.8920,862.3084,28
2,2023-03,56933.9090,693.4499,72
3,2023-04,28295.3450,3488.8352,66
4,2023-05,26319.7670,3196.3918,72
5,2023-06,34669.4796,4999.7594,67
6,2023-07,33946.3930,-841.4826,65
7,2023-08,28918.3385,5765.2250,75
8,2023-09,82670.4288,8593.6302,134
9,2023-10,32413.3390,3469.1749,81


In [57]:
def subcategory_performance(data):
    """
    Generate sub-category sales and profitability analysis.
    """
    result = (
        data.groupby("Sub-Category")
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("Order ID", "nunique")
        )
        .reset_index()
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Profit", ascending=False)

In [58]:
subcategory_result = subcategory_performance(df)

subcategory_result

,Sub-Category,Sales,Profit,Quantity,Orders,Profit Margin (%)
6,Copiers,150745.2900,56093.9365,242,70,37.21
13,Phones,331842.6400,45050.8265,3357,826,13.58
0,Accessories,167380.3180,41936.6357,2976,718,25.05
12,Paper,79540.5380,34511.5070,5207,1205,43.39
3,Binders,207354.8810,31426.1003,6071,1339,15.16
5,Chairs,335768.2490,27223.5323,2431,591,8.11
14,Storage,224644.5540,21285.1115,3191,787,9.48
1,Appliances,108213.1850,18329.4844,1750,459,16.94
9,Furnishings,95598.1260,13891.7430,3799,919,14.53
7,Envelopes,16528.3620,6988.0247,914,251,42.28


In [59]:
def discount_performance(data):
    """
    Analyze the relationship between discount levels and profitability.
    """
    result = (
        data.groupby("Discount")
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("Order ID", "nunique")
        )
        .reset_index()
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Discount")

In [60]:
discount_result = discount_performance(df)

discount_result

,Discount,Sales,Profit,Quantity,Orders,Profit Margin (%)
0,0.00,1.105324e+06,326718.5872,18770,2714,29.56
1,0.10,5.495250e+04,9099.9700,377,91,16.56
2,0.15,2.755852e+04,1418.9915,198,51,5.15
3,0.20,7.739394e+05,91079.9517,13856,2440,11.77
4,0.30,1.044741e+05,-10513.4456,855,214,-10.06
5,0.32,1.449346e+04,-2391.1377,105,27,-16.50
6,0.40,1.164978e+05,-23086.3742,789,186,-19.82
7,0.45,5.484974e+03,-2493.1111,45,10,-45.45
8,0.50,5.891854e+04,-20506.4281,241,64,-34.80
9,0.60,7.105940e+03,-6164.3472,554,138,-86.75


In [61]:
def product_performance(data):
    """
    Generate product-level sales and profitability analysis.
    """
    result = (
        data.groupby(["Product ID", "Product Name"])
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("Order ID", "nunique"),
            Avg_Discount=("Discount", "mean")
        )
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Profit", ascending=False)

In [62]:
product_result = product_performance(df)

product_result.head(10)

,,Sales,Profit,Quantity,Orders,Avg_Discount,Profit Margin (%)
Product ID,Product Name,,,,,,
TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61599.824,25199.9280,20,5,0.120000,40.91
OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,27453.384,7753.0390,31,10,0.240000,28.24
TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,18839.686,6983.8836,38,8,0.200000,37.07
TEC-CO-10003763,Canon PC1060 Personal Laser Copier,11619.834,4570.9347,19,4,0.150000,39.34
TEC-MA-10001127,"HP Designjet T520 Inkjet Large Format Printer - 24"" Color",18374.895,4094.9766,12,3,0.166667,22.29
TEC-MA-10003979,Ativa V4110MDD Micro-Cut Shredder,7699.890,3772.9461,11,2,0.000000,49.00
TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",14299.890,3717.9714,11,2,0.000000,26.00
TEC-AC-10002049,Plantronics Savi W720 Multi-Device Wireless Headset System,9367.290,3696.2820,24,7,0.057143,39.46
OFF-BI-10001120,Ibico EPK-21 Electric Binding System,15875.916,3345.2823,13,3,0.333333,21.07


In [63]:
def export_analysis(result, file_path):
    """
    Export an analysis DataFrame to CSV.
    """
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)

    result.to_csv(file_path, index=False)

    print(f"Exported successfully: {file_path.resolve()}")

In [64]:
REPORTS_DIR = PROJECT_ROOT / "reports"

export_analysis(
    category_result,
    REPORTS_DIR / "category_performance.csv"
)

export_analysis(
    region_result,
    REPORTS_DIR / "region_performance.csv"
)

export_analysis(
    monthly_result,
    REPORTS_DIR / "monthly_performance.csv"
)

print("Analysis exports completed successfully!")

Exported successfully: C:\Users\User\OneDrive\Pictures\Desktop\Shanmukha\Projects\Sales-Intelligence-Analytics\reports\category_performance.csv
Exported successfully: C:\Users\User\OneDrive\Pictures\Desktop\Shanmukha\Projects\Sales-Intelligence-Analytics\reports\region_performance.csv
Exported successfully: C:\Users\User\OneDrive\Pictures\Desktop\Shanmukha\Projects\Sales-Intelligence-Analytics\reports\monthly_performance.csv
Analysis exports completed successfully!


In [65]:
def customer_performance(data):
    """
    Generate customer-level sales and profitability analysis.
    """
    result = (
        data.groupby(["Customer ID", "Customer Name"])
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Orders=("Order ID", "nunique"),
            Quantity=("Quantity", "sum"),
            Avg_Discount=("Discount", "mean")
        )
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Sales", ascending=False)

In [66]:
customer_result = customer_performance(df)

customer_result.head(10)

,,Sales,Profit,Orders,Quantity,Avg_Discount,Profit Margin (%)
Customer ID,Customer Name,,,,,,
SM-20320,Sean Miller,25043.050,-1980.7393,5,50,0.246667,-7.91
TC-20980,Tamara Chand,19052.218,8981.3239,5,42,0.116667,47.14
RB-19360,Raymond Buch,15117.339,6976.0959,6,71,0.094444,46.15
TA-21385,Tom Ashbrook,14595.620,4703.7883,4,36,0.080000,32.23
AB-10105,Adrian Barton,14473.571,5444.8055,10,73,0.240000,37.62
KL-16645,Ken Lonsdale,14175.229,806.8550,12,113,0.200000,5.69
SC-20095,Sanjit Chand,14142.334,5757.4119,9,87,0.063636,40.71
HL-15040,Hunter Lopez,12873.298,5622.4292,6,50,0.018182,43.68
SE-20110,Sanjit Engle,12209.438,2650.6769,11,78,0.110526,21.71


In [67]:
def get_top_products(data, n=10):
    """
    Return the most profitable products.
    """
    result = product_performance(data)

    return result.head(n)

In [68]:
top_products_result = get_top_products(df, 10)

top_products_result

,,Sales,Profit,Quantity,Orders,Avg_Discount,Profit Margin (%)
Product ID,Product Name,,,,,,
TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61599.824,25199.9280,20,5,0.120000,40.91
OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,27453.384,7753.0390,31,10,0.240000,28.24
TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,18839.686,6983.8836,38,8,0.200000,37.07
TEC-CO-10003763,Canon PC1060 Personal Laser Copier,11619.834,4570.9347,19,4,0.150000,39.34
TEC-MA-10001127,"HP Designjet T520 Inkjet Large Format Printer - 24"" Color",18374.895,4094.9766,12,3,0.166667,22.29
TEC-MA-10003979,Ativa V4110MDD Micro-Cut Shredder,7699.890,3772.9461,11,2,0.000000,49.00
TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",14299.890,3717.9714,11,2,0.000000,26.00
TEC-AC-10002049,Plantronics Savi W720 Multi-Device Wireless Headset System,9367.290,3696.2820,24,7,0.057143,39.46
OFF-BI-10001120,Ibico EPK-21 Electric Binding System,15875.916,3345.2823,13,3,0.333333,21.07


In [69]:
def get_loss_making_products(data, n=10):
    """
    Return the most loss-making products.
    """
    result = product_performance(data)

    return (
        result[result["Profit"] < 0]
        .sort_values("Profit")
        .head(n)
    )

In [70]:
loss_products_result = get_loss_making_products(df, 10)

loss_products_result

,,Sales,Profit,Quantity,Orders,Avg_Discount,Profit Margin (%)
Product ID,Product Name,,,,,,
TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,11099.963,-8879.9704,9,3,0.533333,-80.00
TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,16829.901,-4589.9730,18,4,0.400000,-27.27
TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,7999.980,-3839.9904,4,1,0.500000,-48.00
FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tables & Bases,9917.640,-2876.1156,27,5,0.280000,-29.00
FUR-TA-10001889,Bush Advantage Collection Racetrack Conference Table,9544.725,-1934.3976,33,7,0.350000,-20.27
OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,17965.068,-1878.1662,27,6,0.450000,-10.45
TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferencing Unit,22638.480,-1811.0784,6,1,0.500000,-8.00
OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Opener,16656.200,-1299.1836,22,6,0.100000,-7.80
FUR-TA-10001950,Balt Solid Wood Round Tables,6518.754,-1201.0581,19,4,0.200000,-18.42


In [71]:
def get_top_customers_by_profit(data, n=10):
    """
    Return customers ranked by total profit.
    """
    result = customer_performance(data)

    return (
        result
        .sort_values("Profit", ascending=False)
        .head(n)
    )

In [72]:
top_profit_customers_result = get_top_customers_by_profit(df, 10)

top_profit_customers_result

,,Sales,Profit,Orders,Quantity,Avg_Discount,Profit Margin (%)
Customer ID,Customer Name,,,,,,
TC-20980,Tamara Chand,19052.218,8981.3239,5,42,0.116667,47.14
RB-19360,Raymond Buch,15117.339,6976.0959,6,71,0.094444,46.15
SC-20095,Sanjit Chand,14142.334,5757.4119,9,87,0.063636,40.71
HL-15040,Hunter Lopez,12873.298,5622.4292,6,50,0.018182,43.68
AB-10105,Adrian Barton,14473.571,5444.8055,10,73,0.240000,37.62
TA-21385,Tom Ashbrook,14595.620,4703.7883,4,36,0.080000,32.23
CM-12385,Christopher Martinez,8954.020,3899.8904,4,34,0.120000,43.55
KD-16495,Keith Dawkins,8181.256,3038.6254,12,84,0.087500,37.14
AR-10540,Andy Reiter,6608.448,2884.6208,6,33,0.066667,43.65


In [73]:
def save_processing_module():
    """
    Save reusable data-processing functions to src/data_processing.py.
    """
    source_code = '''
from pathlib import Path
import pandas as pd


def load_data(file_path):
    """Load the Superstore dataset."""
    return pd.read_excel(file_path)


def calculate_kpis(data):
    """Calculate key business KPIs."""
    total_sales = data["Sales"].sum()
    total_profit = data["Profit"].sum()
    total_quantity = data["Quantity"].sum()
    total_orders = data["Order ID"].nunique()
    total_customers = data["Customer ID"].nunique()

    profit_margin = (
        total_profit / total_sales * 100
        if total_sales != 0 else 0
    )

    average_order_value = (
        total_sales / total_orders
        if total_orders != 0 else 0
    )

    return {
        "Total Sales": total_sales,
        "Total Profit": total_profit,
        "Total Quantity": total_quantity,
        "Total Orders": total_orders,
        "Total Customers": total_customers,
        "Profit Margin (%)": profit_margin,
        "Average Order Value": average_order_value
    }


def category_performance(data):
    """Generate category-level analysis."""
    result = (
        data.groupby("Category")
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("Order ID", "nunique")
        )
        .reset_index()
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Sales", ascending=False)


def region_performance(data):
    """Generate region-level analysis."""
    result = (
        data.groupby("Region")
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("Order ID", "nunique")
        )
        .reset_index()
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Sales", ascending=False)


def monthly_performance(data):
    """Generate monthly performance analysis."""
    result = (
        data.groupby(data["Order Date"].dt.to_period("M"))
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Orders=("Order ID", "nunique")
        )
        .reset_index()
    )

    result["Order Date"] = result["Order Date"].astype(str)

    return result


def subcategory_performance(data):
    """Generate sub-category analysis."""
    result = (
        data.groupby("Sub-Category")
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("Order ID", "nunique")
        )
        .reset_index()
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Profit", ascending=False)


def discount_performance(data):
    """Generate discount profitability analysis."""
    result = (
        data.groupby("Discount")
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("Order ID", "nunique")
        )
        .reset_index()
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Discount")


def product_performance(data):
    """Generate product-level analysis."""
    result = (
        data.groupby(["Product ID", "Product Name"])
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("Order ID", "nunique"),
            Avg_Discount=("Discount", "mean")
        )
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Profit", ascending=False)


def customer_performance(data):
    """Generate customer-level analysis."""
    result = (
        data.groupby(["Customer ID", "Customer Name"])
        .agg(
            Sales=("Sales", "sum"),
            Profit=("Profit", "sum"),
            Orders=("Order ID", "nunique"),
            Quantity=("Quantity", "sum"),
            Avg_Discount=("Discount", "mean")
        )
    )

    result["Profit Margin (%)"] = (
        result["Profit"] / result["Sales"] * 100
    ).round(2)

    return result.sort_values("Sales", ascending=False)


def get_top_products(data, n=10):
    """Return the most profitable products."""
    return product_performance(data).head(n)


def get_loss_making_products(data, n=10):
    """Return the most loss-making products."""
    result = product_performance(data)

    return (
        result[result["Profit"] < 0]
        .sort_values("Profit")
        .head(n)
    )


def get_top_customers_by_profit(data, n=10):
    """Return customers ranked by profit."""
    return (
        customer_performance(data)
        .sort_values("Profit", ascending=False)
        .head(n)
    )


def export_analysis(result, file_path):
    """Export an analysis DataFrame to CSV."""
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(file_path, index=False)
'''

    module_path = PROJECT_ROOT / "src" / "data_processing.py"
    module_path.parent.mkdir(parents=True, exist_ok=True)

    module_path.write_text(source_code.strip(), encoding="utf-8")

    print("Module created successfully!")
    print("Location:", module_path)

In [74]:
save_processing_module()

module_path = PROJECT_ROOT / "src" / "data_processing.py"

print("File exists:", module_path.exists())
print("File size:", module_path.stat().st_size, "bytes")

Module created successfully!
Location: C:\Users\User\OneDrive\Pictures\Desktop\Shanmukha\Projects\Sales-Intelligence-Analytics\src\data_processing.py
File exists: True
File size: 5421 bytes


In [75]:
import sys

src_path = str(PROJECT_ROOT / "src")

if src_path not in sys.path:
    sys.path.append(src_path)

print("src path added successfully!")

src path added successfully!


In [76]:
import data_processing

print("data_processing module imported successfully!")

data_processing module imported successfully!


In [77]:
processed_df = data_processing.load_data(DATA_FILE)

print("Data loaded through reusable module!")
print("Shape:", processed_df.shape)

Data loaded through reusable module!
Shape: (10194, 21)


In [78]:
module_kpis = data_processing.calculate_kpis(processed_df)

print("BUSINESS KPIs")
print("=" * 40)

for key, value in module_kpis.items():
    print(f"{key}: {value:,.2f}")

BUSINESS KPIs
Total Sales: 2,326,534.35
Total Profit: 292,296.81
Total Quantity: 38,654.00
Total Orders: 5,111.00
Total Customers: 804.00
Profit Margin (%): 12.56
Average Order Value: 455.20


In [79]:
module_category = data_processing.category_performance(processed_df)

module_category

,Category,Sales,Profit,Quantity,Orders,Profit Margin (%)
2,Technology,839893.2790,146543.3756,7017,1560,17.45
0,Furniture,754747.7613,19729.9956,8369,1819,2.61
1,Office Supplies,731893.3140,126023.4434,23268,3812,17.22


In [80]:
module_region = data_processing.region_performance(processed_df)

module_region

,Region,Sales,Profit,Quantity,Orders,Profit Margin (%)
3,West,739813.6085,110798.8170,12466,1635,14.98
1,East,691828.1680,94883.2603,11159,1475,13.71
0,Central,503170.6728,39865.3070,8820,1179,7.92
2,South,391721.9050,46749.4303,6209,822,11.93


In [81]:
module_monthly = data_processing.monthly_performance(processed_df)

module_monthly.head(12)

,Order Date,Sales,Profit,Orders
0,2023-01,14518.0550,2539.3907,33
1,2023-02,4519.8920,862.3084,28
2,2023-03,56933.9090,693.4499,72
3,2023-04,28295.3450,3488.8352,66
4,2023-05,26319.7670,3196.3918,72
5,2023-06,34669.4796,4999.7594,67
6,2023-07,33946.3930,-841.4826,65
7,2023-08,28918.3385,5765.2250,75
8,2023-09,82670.4288,8593.6302,134
9,2023-10,32413.3390,3469.1749,81


In [82]:
module_subcategory = data_processing.subcategory_performance(processed_df)

module_subcategory.head(10)

,Sub-Category,Sales,Profit,Quantity,Orders,Profit Margin (%)
6,Copiers,150745.290,56093.9365,242,70,37.21
13,Phones,331842.640,45050.8265,3357,826,13.58
0,Accessories,167380.318,41936.6357,2976,718,25.05
12,Paper,79540.538,34511.5070,5207,1205,43.39
3,Binders,207354.881,31426.1003,6071,1339,15.16
5,Chairs,335768.249,27223.5323,2431,591,8.11
14,Storage,224644.554,21285.1115,3191,787,9.48
1,Appliances,108213.185,18329.4844,1750,459,16.94
9,Furnishings,95598.126,13891.7430,3799,919,14.53
7,Envelopes,16528.362,6988.0247,914,251,42.28


In [83]:
module_products = data_processing.get_top_products(
    processed_df,
    10
)

module_products

,,Sales,Profit,Quantity,Orders,Avg_Discount,Profit Margin (%)
Product ID,Product Name,,,,,,
TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61599.824,25199.9280,20,5,0.120000,40.91
OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,27453.384,7753.0390,31,10,0.240000,28.24
TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,18839.686,6983.8836,38,8,0.200000,37.07
TEC-CO-10003763,Canon PC1060 Personal Laser Copier,11619.834,4570.9347,19,4,0.150000,39.34
TEC-MA-10001127,"HP Designjet T520 Inkjet Large Format Printer - 24"" Color",18374.895,4094.9766,12,3,0.166667,22.29
TEC-MA-10003979,Ativa V4110MDD Micro-Cut Shredder,7699.890,3772.9461,11,2,0.000000,49.00
TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",14299.890,3717.9714,11,2,0.000000,26.00
TEC-AC-10002049,Plantronics Savi W720 Multi-Device Wireless Headset System,9367.290,3696.2820,24,7,0.057143,39.46
OFF-BI-10001120,Ibico EPK-21 Electric Binding System,15875.916,3345.2823,13,3,0.333333,21.07


In [84]:
module_loss_products = data_processing.get_loss_making_products(
    processed_df,
    10
)

module_loss_products

,,Sales,Profit,Quantity,Orders,Avg_Discount,Profit Margin (%)
Product ID,Product Name,,,,,,
TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,11099.963,-8879.9704,9,3,0.533333,-80.00
TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,16829.901,-4589.9730,18,4,0.400000,-27.27
TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,7999.980,-3839.9904,4,1,0.500000,-48.00
FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tables & Bases,9917.640,-2876.1156,27,5,0.280000,-29.00
FUR-TA-10001889,Bush Advantage Collection Racetrack Conference Table,9544.725,-1934.3976,33,7,0.350000,-20.27
OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,17965.068,-1878.1662,27,6,0.450000,-10.45
TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferencing Unit,22638.480,-1811.0784,6,1,0.500000,-8.00
OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Opener,16656.200,-1299.1836,22,6,0.100000,-7.80
FUR-TA-10001950,Balt Solid Wood Round Tables,6518.754,-1201.0581,19,4,0.200000,-18.42


In [85]:
module_customers = data_processing.customer_performance(
    processed_df
)

module_customers.head(10)

,,Sales,Profit,Orders,Quantity,Avg_Discount,Profit Margin (%)
Customer ID,Customer Name,,,,,,
SM-20320,Sean Miller,25043.050,-1980.7393,5,50,0.246667,-7.91
TC-20980,Tamara Chand,19052.218,8981.3239,5,42,0.116667,47.14
RB-19360,Raymond Buch,15117.339,6976.0959,6,71,0.094444,46.15
TA-21385,Tom Ashbrook,14595.620,4703.7883,4,36,0.080000,32.23
AB-10105,Adrian Barton,14473.571,5444.8055,10,73,0.240000,37.62
KL-16645,Ken Lonsdale,14175.229,806.8550,12,113,0.200000,5.69
SC-20095,Sanjit Chand,14142.334,5757.4119,9,87,0.063636,40.71
HL-15040,Hunter Lopez,12873.298,5622.4292,6,50,0.018182,43.68
SE-20110,Sanjit Engle,12209.438,2650.6769,11,78,0.110526,21.71


In [86]:
module_top_profit_customers = (
    data_processing.get_top_customers_by_profit(
        processed_df,
        10
    )
)

module_top_profit_customers

,,Sales,Profit,Orders,Quantity,Avg_Discount,Profit Margin (%)
Customer ID,Customer Name,,,,,,
TC-20980,Tamara Chand,19052.218,8981.3239,5,42,0.116667,47.14
RB-19360,Raymond Buch,15117.339,6976.0959,6,71,0.094444,46.15
SC-20095,Sanjit Chand,14142.334,5757.4119,9,87,0.063636,40.71
HL-15040,Hunter Lopez,12873.298,5622.4292,6,50,0.018182,43.68
AB-10105,Adrian Barton,14473.571,5444.8055,10,73,0.240000,37.62
TA-21385,Tom Ashbrook,14595.620,4703.7883,4,36,0.080000,32.23
CM-12385,Christopher Martinez,8954.020,3899.8904,4,34,0.120000,43.55
KD-16495,Keith Dawkins,8181.256,3038.6254,12,84,0.087500,37.14
AR-10540,Andy Reiter,6608.448,2884.6208,6,33,0.066667,43.65


In [87]:
module_discount = data_processing.discount_performance(
    processed_df
)

module_discount

,Discount,Sales,Profit,Quantity,Orders,Profit Margin (%)
0,0.00,1.105324e+06,326718.5872,18770,2714,29.56
1,0.10,5.495250e+04,9099.9700,377,91,16.56
2,0.15,2.755852e+04,1418.9915,198,51,5.15
3,0.20,7.739394e+05,91079.9517,13856,2440,11.77
4,0.30,1.044741e+05,-10513.4456,855,214,-10.06
5,0.32,1.449346e+04,-2391.1377,105,27,-16.50
6,0.40,1.164978e+05,-23086.3742,789,186,-19.82
7,0.45,5.484974e+03,-2493.1111,45,10,-45.45
8,0.50,5.891854e+04,-20506.4281,241,64,-34.80
9,0.60,7.105940e+03,-6164.3472,554,138,-86.75


In [88]:
print("Validation Summary")
print("=" * 40)

print("Rows:", len(processed_df))
print("Columns:", len(processed_df.columns))
print("Missing values:", processed_df.isna().sum().sum())
print("Duplicate rows:", processed_df.duplicated().sum())

Validation Summary
Rows: 10194
Columns: 21
Missing values: 0
Duplicate rows: 0


In [89]:
expected_rows = 10194
expected_columns = 21

assert len(processed_df) == expected_rows
assert len(processed_df.columns) == expected_columns

print("Dataset structure validation: PASSED")

Dataset structure validation: PASSED


In [90]:
assert processed_df["Order ID"].nunique() == 5111
assert processed_df["Customer ID"].nunique() == 804
assert processed_df["Product ID"].nunique() == 1862

print("Business entity validation: PASSED")

Business entity validation: PASSED


In [91]:
assert round(module_kpis["Total Sales"], 2) == 2326534.35
assert round(module_kpis["Total Profit"], 2) == 292296.81

print("KPI validation: PASSED")

KPI validation: PASSED


In [92]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

data_processing.export_analysis(
    module_customers.reset_index(),
    REPORTS_DIR / "customer_performance.csv"
)

data_processing.export_analysis(
    module_subcategory,
    REPORTS_DIR / "subcategory_performance.csv"
)

data_processing.export_analysis(
    module_discount,
    REPORTS_DIR / "discount_performance.csv"
)

print("Additional analysis reports exported!")

Additional analysis reports exported!


In [93]:
generated_files = sorted(
    REPORTS_DIR.glob("*.csv")
)

print("Generated report files:")
for file in generated_files:
    print("-", file.name)

print("\nTotal CSV reports:", len(generated_files))

Generated report files:
- category_performance.csv
- customer_performance.csv
- discount_performance.csv
- monthly_performance.csv
- region_performance.csv
- subcategory_performance.csv

Total CSV reports: 6


In [94]:
module_text = module_path.read_text(
    encoding="utf-8"
)

print("Final module validation")
print("=" * 40)
print("Module exists:", module_path.exists())
print("Module size:", module_path.stat().st_size, "bytes")
print("Contains load_data:", "def load_data" in module_text)
print("Contains calculate_kpis:", "def calculate_kpis" in module_text)
print("Contains product_performance:", "def product_performance" in module_text)
print("Contains customer_performance:", "def customer_performance" in module_text)

print("\nData processing module completed successfully! ✅")

Final module validation
Module exists: True
Module size: 5421 bytes
Contains load_data: True
Contains calculate_kpis: True
Contains product_performance: True
Contains customer_performance: True

Data processing module completed successfully! ✅
